# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [13]:
import os
import numpy as np
import importlib.util
import pickle
from scipy.sparse import coo_matrix
from spektral.data import Graph


def load_params(py_path):
    """
    Dynamically loads a Python file containing parameters, e.g.:
        v0 = [0.1, 1.3]
        W = [[0.0, 0.08],
             [0.08, 0.0]]
        A0 = [0.9, 0.9]
        P0 = [3.812, 3.812]
        Dr = 50
        kappa_A = 0.4
        kappa_P = 0.07
        a = 0.25
        k = 2.5
    """
    print(f"> Attempting to load parameters from: {py_path}")
    spec = importlib.util.spec_from_file_location("param_module", py_path)
    param_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(param_module)
    
    # Gather the parameters into a dictionary:
    p = {
        "v0": getattr(param_module, "v0", None),
        "W": getattr(param_module, "W", None),
        "A0": getattr(param_module, "A0", None),
        "P0": getattr(param_module, "P0", None),
        "Dr": getattr(param_module, "Dr", None),
        "kappa_A": getattr(param_module, "kappa_A", None),
        "kappa_P": getattr(param_module, "kappa_P", None),
        "a": getattr(param_module, "a", None),
        "k": getattr(param_module, "k", None),
    }
    print(f"> Loaded parameters: {p}")
    return p


def build_dataset(
    data_dir: str,
    param_path: str,
    t_start: int = 0,
    t_end: int = 200,
    t_step: int = 1,
):
    """
    Build a dataset of Graph snapshots (G_t) from time t in [t_start, t_end],
    stepping by t_step. For each t, we also produce the adjacency at t+1 (if 
    available) for training a GNN that predicts adjacency changes from t -> t+1.

    Key difference: we only gather "unique" edges (i<j), then duplicate them 
    to produce 2E edges in a well-defined forward-block + reverse-block.

    Returns:
      A list of (Graph, adjacency_{t+1}) for times t.
    """
    print(f"\n=== build_dataset ===")
    print(f"Data dir  = {data_dir}")
    print(f"Param file= {param_path}")
    print(f"Time range= [{t_start}, {t_end}] with step={t_step}\n")

    # 1) Load the param file
    p = load_params(param_path)
    
    # 2) We'll store a list of (G_t, adjacency_{t+1}) for each valid t
    dataset = []

    # 3) Iterate over time steps
    t_list = range(t_start, t_end + 1, t_step)
    print(f">> Will look for data_{{t}}.npy from t in {list(t_list)}")

    for t in t_list:
        data_file_t = os.path.join(data_dir, f"data_{t}.npy")
        if not os.path.exists(data_file_t):
            print(f" - data_{t}.npy not found. Skipping.")
            continue

        print(f"\nLoading timepoint t={t} from {data_file_t}")
        data_t = np.load(data_file_t, allow_pickle=True).item()

        # We'll only build a label adjacency if (t + t_step) also exists
        t_next = t + t_step
        data_file_tplus = os.path.join(data_dir, f"data_{t_next}.npy")
        if os.path.exists(data_file_tplus):
            data_tplus = np.load(data_file_tplus, allow_pickle=True).item()
            adj_label = data_tplus["cell_adj"]  # shape [n_c, n_c]
            print(f"   Found adjacency label at t+{t_step} = {t_next}")
        else:
            adj_label = None
            print(f"   No data for t+{t_step} = {t_next}. Label is None.")

        # Extract your raw data from dictionary:
        cell_x    = data_t["cell_x"]           # shape [n_c, 2]
        cell_type = data_t["cell_type"]        # shape [n_c]
        area      = data_t["area"]             # shape [n_c]
        perimeter = data_t["perimeter"]        # shape [n_c]
        cell_adj  = data_t["cell_adj"]         # shape [n_c, n_c]

        # "edge_voronoi_length" might or might not exist
        voronoi_len = data_t.get("edge_voronoi_length", None)
        if voronoi_len is None:
            print("   edge_voronoi_length not found in data dict.")
        else:
            print("   Found edge_voronoi_length in data dict.")

        n_c = cell_x.shape[0]
        print(f"   n_c = {n_c} cells in this timepoint")

        # 4) Build node features for time t
        #    Example: [ x_i, y_i, area_i, perimeter_i, cell_type_i, 
        #               A0(type), P0(type), Dr, kappa_A, kappa_P, a, k ]
        node_feats = []
        for i in range(n_c):
            ctype = cell_type[i]
            A0_val = p["A0"][ctype]
            P0_val = p["P0"][ctype]
            (x_i, y_i) = cell_x[i]
            feats = [
                x_i,
                y_i,
                area[i],
                perimeter[i],
                float(ctype),
                A0_val,
                P0_val,
                p["Dr"],
                p["kappa_A"],
                p["kappa_P"],
                p["a"],
                p["k"],
            ]
            node_feats.append(feats)
        node_feats = np.array(node_feats, dtype=np.float32)  # shape [n_c, D]

        # 5) Build adjacency + edge features, but first we gather only "unique" edges (i<j)
        unique_rows = []
        unique_cols = []
        unique_edge_feats = []

        for i in range(n_c):
            for j in range(i+1, n_c):
                if cell_adj[i, j] == 1:
                    # i < j, this is the unique edge
                    unique_rows.append(i)
                    unique_cols.append(j)
                    
                    # Voronoi length if available
                    vlen = 0.0
                    if voronoi_len is not None:
                        vlen = voronoi_len[i, j]
                    
                    # W_interaction for this pair
                    ctype_i = cell_type[i]
                    ctype_j = cell_type[j]
                    Wij = p["W"][ctype_i][ctype_j]
                    
                    # Build the edge feature for i->j
                    unique_edge_feats.append([vlen, Wij])

        E_unique = len(unique_rows)  # number of unique edges

        # 6) Duplicate them to get forward + reverse
        rows_duplicated = unique_rows + unique_cols  # length 2*E_unique
        cols_duplicated = unique_cols + unique_rows  # same length
        edge_feats_duplicated = unique_edge_feats + unique_edge_feats  # also length 2*E_unique

        row_ar = np.array(rows_duplicated, dtype=np.int32)
        col_ar = np.array(cols_duplicated, dtype=np.int32)
        edge_feats_ar = np.array(edge_feats_duplicated, dtype=np.float32)

        data_ar = np.ones_like(row_ar, dtype=np.float32)  # or whatever weight you want
        a_coo = coo_matrix((data_ar, (row_ar, col_ar)), shape=(n_c, n_c))

        # 7) Construct the Spektral Graph for time t
        #    shape of edge_feats_ar => [2*E_unique, D_edge]
        G_t = Graph(
            x=node_feats,
            a=a_coo,
            e=edge_feats_ar
        )
        
        # 8) Append (G_t, adj_label) to dataset
        dataset.append((G_t, adj_label))

    print(f"\nBuild complete. Total {len(dataset)} snapshots processed.")
    return dataset

In [14]:

if __name__ == "__main__":
    # Example usage with your lists of data_dirs and param_files.
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I/matrix_output_{i}"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    # We'll define some time config:
    T_START = 0
    T_END = 2000   # up to 2000
    T_STEP = 200     # maybe your data are spaced in increments of 4

    # We'll build dataset for each sim ID in turn
    all_datasets = []
    for sim_idx in range(10):
        ddir = data_dirs[sim_idx]
        pfile = param_files[sim_idx]
        ds = build_dataset(
            data_dir=ddir,
            param_path=pfile,
            t_start=T_START,
            t_end=T_END,
            t_step=T_STEP
        )
        print(
            f"Simulation ID = {sim_idx+1}: built dataset with {len(ds)} steps "
            f"from directory '{ddir}'."
        )
        all_datasets.append(ds)

    # Now save `all_datasets` to a local file (e.g., "all_datasets.pkl"):

    save_path = "all_datasets.pkl"
    print(f"\nSaving all_datasets to {save_path} ...")
    with open(save_path, "wb") as f:
        pickle.dump(all_datasets, f)

    print("Done. The file contains a list of 10 datasets (one per simulation).")

    # Now all_datasets is a list of datasets, each one being a list of (Graph, adj_label).
    # all_datasets[i] corresponds to simulation i+1.
    
    # Example: just checking the first step from simulation #1
    if len(all_datasets[0]) > 0:
        graph_0, adj_label_0 = all_datasets[0][0]
        print("First Graph node features shape:", graph_0.x.shape)
        print("First Graph adjacency shape:", graph_0.a.shape)
        print("First Graph edge feature shape:", graph_0.e.shape)
        if adj_label_0 is not None:
            print("Next adjacency shape:", adj_label_0.shape)



=== build_dataset ===
Data dir  = /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I/matrix_output_1
Param file= /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Time range= [0, 2000] with step=200

> Attempting to load parameters from: /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
> Loaded parameters: {'v0': [0.1, 1.3], 'W': ([0.0, 0.08], [0.08, 0.0]), 'A0': [0.9, 0.9], 'P0': [3.812, 3.812], 'Dr': 50, 'kappa_A': 0.4, 'kappa_P': 0.07, 'a': 0.25, 'k': 2.5}
>> Will look for data_{t}.npy from t in [0, 200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]

Loading timepoint t=0 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I/matrix_output_1/data_0.npy
   Found adjacency label at t+200 = 200
   Found edge_voronoi_length in data dict.
   n_c = 840 cells in this timepoint

Loading timepoint t=200 from /Users/sophia01px2019/Downloads/gartner_lab_rotati

Here is where we:

Load all_datasets.pkl.
For each (Graph_t, adj_label_{t+1}), convert to PyG format.
Use NeighborLoader to sample subgraphs for each node (or a fraction of them).
Build next-step labels (adj, area, perimeter, boundary length, etc.) restricted to subgraph nodes.
Save subgraph+label to some file for training.

In [3]:
import torch_scatter
import torch_sparse
import torch_cluster
import torch_spline_conv
import torch_geometric

In [20]:
import os
import pickle
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader

def convert_spektral_to_pyg(spektral_graph):
    """
    Convert a spektral Graph object to PyG Data. 
    Returns Data(x=..., edge_index=..., edge_attr=..., <other>).
    """

    x = torch.FloatTensor(spektral_graph.x)  # shape [N, D_node]
    # spektral_graph.a is a coo_matrix
    row = spektral_graph.a.row
    col = spektral_graph.a.col
    edge_index = np.vstack([row, col])  # shape [2, E]
    edge_index_t = torch.LongTensor(edge_index)

    edge_attr = torch.FloatTensor(spektral_graph.e) if spektral_graph.e is not None else None

    pyg_data = Data(
        x=x,
        edge_index=edge_index_t,
        edge_attr=edge_attr,
    )
    return pyg_data


def build_subgraph_label(
    subgraph_data, 
    adjacency_tplus,  # shape [N, N], next-step adjacency
    node_id_mapping,  # local->global
    x_time_t,         # shape [N, D_node], node features from *time t* (for constants)
    x_time_tplus,     # shape [N, D_node], node features from *time t+1* (for predicted)
    e_time_t=None,    # shape [2E, D_edge], edge feats (time t) if storing constants like W
    row_t=None,
    col_t=None,
    e_time_tplus=None,# shape [2E, D_edge], edge feats (time t+1) for boundary length
    row_tplus=None,
    col_tplus=None
):
    """
    Builds a label dictionary for the subgraph, including:
      1) node_constants: time-invariant fields from x_time_t (cell_type, etc.)
      2) edge_constants: time-invariant fields from e_time_t (like W).
      3) node_labels: next-step geometry (area, perimeter, coords) from x_time_tplus.
      4) edge_exists: adjacency at t+1
      5) edge_length: boundary length at t+1 (if e_time_tplus is provided)

    subgraph_data.edge_index => local edges; subgraph_data.n_id => global node IDs

    NOTE: We have removed the "+ E_unique_tp" indexing offset to avoid out-of-range errors.
    We do direct searching in row_tplus, col_tplus if needed.
    """
    n_sub = subgraph_data.num_nodes
    E_sub = subgraph_data.edge_index.shape[1]

    # --------------------------------------------------------
    # 1) Extract constant node features from x_time_t
    #    Suppose in x_time_t columns we have:
    #       [0->x, 1->y, 2->area, 3->perimeter, 4->cell_type, 5->A0, 6->P0, 7->Dr, 8->kappaA, 9->kappaP, 10->a, 11->k, ...]
    #    We only store the "constant" parts: cell_type, A0, P0, Dr, ...
    # --------------------------------------------------------
    node_constants = {
        "cell_type": np.zeros(n_sub, dtype=np.float32),
        "A0":        np.zeros(n_sub, dtype=np.float32),
        "P0":        np.zeros(n_sub, dtype=np.float32),
        "Dr":        np.zeros(n_sub, dtype=np.float32),
        "kappa_A":   np.zeros(n_sub, dtype=np.float32),
        "kappa_P":   np.zeros(n_sub, dtype=np.float32),
        "a":         np.zeros(n_sub, dtype=np.float32),
        "k":         np.zeros(n_sub, dtype=np.float32),
    }

    # --------------------------------------------------------
    # 2) Extract predicted (time-varying) node features from x_time_tplus
    #    e.g. coords, area, perimeter
    # --------------------------------------------------------
    node_labels = {
        "coords":    np.zeros((n_sub, 2), dtype=np.float32),
        "area":      np.zeros(n_sub, dtype=np.float32),
        "perimeter": np.zeros(n_sub, dtype=np.float32),
    }

    # Build local->global mapping
    edge_index_sub = subgraph_data.edge_index
    for local_i in range(n_sub):
        global_i = node_id_mapping[local_i]
        # (A) Fill node_constants from x_time_t
        node_constants["cell_type"][local_i] = x_time_t[global_i, 4]
        node_constants["A0"][local_i]        = x_time_t[global_i, 5]
        node_constants["P0"][local_i]        = x_time_t[global_i, 6]
        node_constants["Dr"][local_i]        = x_time_t[global_i, 7]
        node_constants["kappa_A"][local_i]   = x_time_t[global_i, 8]
        node_constants["kappa_P"][local_i]   = x_time_t[global_i, 9]
        node_constants["a"][local_i]         = x_time_t[global_i, 10]
        node_constants["k"][local_i]         = x_time_t[global_i, 11]

        # (B) Fill time-varying node labels from x_time_tplus
        node_labels["coords"][local_i, 0] = x_time_tplus[global_i, 0]  # x
        node_labels["coords"][local_i, 1] = x_time_tplus[global_i, 1]  # y
        node_labels["area"][local_i]      = x_time_tplus[global_i, 2]
        node_labels["perimeter"][local_i] = x_time_tplus[global_i, 3]

    # --------------------------------------------------------
    # 3) Edge-level: store "constant" from e_time_t if we have W
    #    (like e_time_t[e_k, 1] => W param?), plus time-varying edge length from e_time_tplus
    # --------------------------------------------------------
    edge_constants = None
    if e_time_t is not None:
        edge_constants = np.zeros((E_sub,), dtype=np.float32)  # store W or similar
    edge_exists = np.zeros(E_sub, dtype=np.float32)
    edge_length = None
    if e_time_tplus is not None:
        edge_length = np.zeros(E_sub, dtype=np.float32)

    E_unique_t = len(row_t) if row_t is not None else 0
    E_unique_tp = len(row_tplus) if row_tplus is not None else 0

    for e_i in range(E_sub):
        local_s = edge_index_sub[0, e_i].item()
        local_t_ = edge_index_sub[1, e_i].item()
        global_s = node_id_mapping[local_s]
        global_t_ = node_id_mapping[local_t_]

        # (A) Time-invariant edge constant from e_time_t => e.g. W param
        if edge_constants is not None:
            found_e_k = False
            for e_k in range(E_unique_t):
                if row_t[e_k] == global_s and col_t[e_k] == global_t_:
                    # e_time_t[e_k, 1] might be W param, e_time_t[e_k, 0] might be boundary length?
                    edge_constants[e_i] = e_time_t[e_k, 1]
                    found_e_k = True
                    break
            if not found_e_k:
                edge_constants[e_i] = 0.0  # or some default

        # (B) Binary adjacency at t+1
        edge_exists[e_i] = adjacency_tplus[global_s, global_t_]

        # (C) If edge_exists > 0, read boundary length from e_time_tplus
        if edge_length is not None and edge_exists[e_i] > 0.5:
            found_e_kp = False
            for e_kp in range(E_unique_tp):
                if (row_tplus[e_kp] == global_s and col_tplus[e_kp] == global_t_) \
                   or (row_tplus[e_kp] == global_t_ and col_tplus[e_kp] == global_s):
                    # Just use e_time_tplus[e_kp, 0], 
                    edge_length[e_i] = e_time_tplus[e_kp, 0]
                    found_e_kp = True
                    break
            if not found_e_kp:
                edge_length[e_i] = 0.0

    # --------------------------------------------------------
    # 4) Combine all into final sub_label_dict
    # --------------------------------------------------------
    sub_label_dict = {
        # Time-invariant
        "node_constants": node_constants,  # cell_type, A0, etc.
        "edge_constants": edge_constants,  # W param
        # Time-varying
        "node_labels": node_labels,        # area, perimeter, coords at t+1
        "edge_exists": edge_exists,        # adjacency at t+1
    }
    if edge_length is not None:
        sub_label_dict["edge_length"] = edge_length

    return sub_label_dict


def neighbor_sampling_for_dataset(
    all_datasets_path,
    out_dir,
    num_neighbors=[10, 10, 10],
    batch_size=1
):
    """
    1) Load all_datasets (list of list of (Graph_t, adj_label_t+1)).
    2) For each time t, convert G_t -> PyG, neighbor sample subgraphs.
    3) Also get G_{t+1} for next-step node/edge features, plus adjacency_label_{t+1}.
    4) For each subgraph, build 'sub_label_dict' containing:
       - time-invariant node/edge features (cell_type, W),
       - next-step node geometry (area, perimeter, coords),
       - adjacency and boundary length for t+1 edges.
    5) Save (subgraph_data, sub_label_dict).
    """
    import torch
    os.makedirs(out_dir, exist_ok=True)

    # ------------------ LOAD DATASETS ------------------
    with open(all_datasets_path, "rb") as f:
        all_datasets = pickle.load(f)

    print(f"Loaded all_datasets from '{all_datasets_path}'. Found {len(all_datasets)} simulations.\n")
    
    for sim_idx, ds in enumerate(all_datasets):
        # ds = [ (G_0, adj_label_1), (G_1, adj_label_2), ..., (G_T, None) ]
        n_timepoints = len(ds)
        print(f"=== Starting simulation #{sim_idx+1} with {n_timepoints} timepoints.")
        
        # We'll store a sample subgraph from the last timepoint's neighbor sampling
        # to print at the end, if we want a shape summary:
        last_subgraph_sample = None

        for t_idx, (spektral_graph_t, adj_label_tplus) in enumerate(ds):
            if adj_label_tplus is None:
                # no adjacency for t+1 => skip
                continue
            if t_idx+1 >= n_timepoints:
                # no G_{t+1}
                continue

            print(f"  Processing simulation={sim_idx+1}, timepoint={t_idx} ...")

            # Retrieve G_{t+1} for node/edge features
            spektral_graph_tplus, _ = ds[t_idx+1]

            # Convert time-t graph to PyG
            pyg_data_t = convert_spektral_to_pyg(spektral_graph_t)

            x_time_t  = spektral_graph_t.x       # shape [N, D_node]
            e_time_t  = spektral_graph_t.e       # shape [2E, D_edge] or None
            row_t     = spektral_graph_t.a.row   if spektral_graph_t.a is not None else None
            col_t     = spektral_graph_t.a.col   if spektral_graph_t.a is not None else None

            x_time_tplus = spektral_graph_tplus.x
            e_time_tplus = spektral_graph_tplus.e
            row_tplus    = spektral_graph_tplus.a.row if spektral_graph_tplus.a is not None else None
            col_tplus    = spektral_graph_tplus.a.col if spektral_graph_tplus.a is not None else None

            adjacency_tplus = adj_label_tplus  # shape [N, N]

            loader = NeighborLoader(
                pyg_data_t,
                num_neighbors=num_neighbors,
                batch_size=batch_size,
                shuffle=False,
            )

            subgraphs_for_t = []
            total_subgraphs = 0
            for step_i, subgraph_data in enumerate(loader):
                # Build local->global mapping
                node_id_mapping = {}
                for local_i in range(subgraph_data.num_nodes):
                    node_id_mapping[local_i] = subgraph_data.n_id[local_i].item()

                # Build subgraph label
                sub_label_dict = build_subgraph_label(
                    subgraph_data,
                    adjacency_tplus,
                    node_id_mapping,
                    x_time_t,           # time-t node feats => constants
                    x_time_tplus,       # time-(t+1) node feats => predicted geometry
                    e_time_t, row_t, col_t,
                    e_time_tplus, row_tplus, col_tplus
                )

                subgraphs_for_t.append((subgraph_data, sub_label_dict))

                # Print a line for each subgraph created
                num_nodes_sub = subgraph_data.num_nodes
                num_edges_sub = subgraph_data.num_edges
                print(f"    Created subgraph #{step_i+1} with {num_nodes_sub} nodes, {num_edges_sub} edges.")

                total_subgraphs += 1
                last_subgraph_sample = (subgraph_data, sub_label_dict)  # store the last one

            # Save
            out_name = f"subgraphs_sim{sim_idx+1}_t{t_idx}.pkl"
            out_path = os.path.join(out_dir, out_name)
            with open(out_path, "wb") as f:
                pickle.dump(subgraphs_for_t, f)

            print(f"  => [Sim {sim_idx+1}, time={t_idx}] Saved {total_subgraphs} subgraphs to {out_path}")

        # At the end of this simulation, let's print an example subgraph shape/features
        if last_subgraph_sample is not None:
            (sub_data, sub_lbl) = last_subgraph_sample
            print(f"\n=== Example subgraph from simulation {sim_idx+1} ===")
            print(f"Nodes in subgraph: {sub_data.num_nodes}, Edges in subgraph: {sub_data.num_edges}")
            print(f"node_constants keys: {list(sub_lbl['node_constants'].keys())}")
            print(f"node_labels keys: {list(sub_lbl['node_labels'].keys())}")
            if sub_lbl["edge_constants"] is not None:
                print(f"edge_constants shape: {sub_lbl['edge_constants'].shape}")
            print(f"edge_exists shape: {sub_lbl['edge_exists'].shape}")
            if "edge_length" in sub_lbl:
                print(f"edge_length shape: {sub_lbl['edge_length'].shape}")
            print("===========================================\n")

    print("All neighbor-sampled subgraphs saved with debug prints.")


In [21]:

if __name__ == "__main__":

    ALL_DATASETS_PATH = "all_datasets.pkl"  
    # file you previously created that holds the list of datasets: 
    #   all_datasets[sim_index] => list of (Graph, adjacency_label)

    OUT_SUBGRAPHS_DIR = "subgraphs_output"  
    # We'll store the neighbor-sampled subgraphs in this directory.

    # If you want to specify neighbor-sampler parameters:
    NUM_NEIGHBORS = [6, 6, 6]  # e.g. 3-layer sampling: 6 neighbors for the first node, then 6 neighbors per each of these 6 1st layer neighbors. If neighbor # < # defined then takes all neighbors. 
    BATCH_SIZE = 1           # If 1, each subgraph is "centered" on a single node per iteration

    neighbor_sampling_for_dataset(
        all_datasets_path=ALL_DATASETS_PATH,   # path to 'all_datasets.pkl'
        out_dir=OUT_SUBGRAPHS_DIR,
        num_neighbors=NUM_NEIGHBORS, # length can be flexibly adjusted to increase or decrease layers
        batch_size=BATCH_SIZE
    )

    print("Neighbor sampling complete.")

Loaded all_datasets from 'all_datasets.pkl'. Found 10 simulations.

=== Starting simulation #1 with 11 timepoints.
  Processing simulation=1, timepoint=0 ...
    Created subgraph #1 with 41 nodes, 118 edges.
    Created subgraph #2 with 36 nodes, 97 edges.
    Created subgraph #3 with 39 nodes, 101 edges.
    Created subgraph #4 with 41 nodes, 121 edges.
    Created subgraph #5 with 39 nodes, 102 edges.
    Created subgraph #6 with 40 nodes, 113 edges.
    Created subgraph #7 with 30 nodes, 78 edges.
    Created subgraph #8 with 40 nodes, 107 edges.
    Created subgraph #9 with 35 nodes, 94 edges.
    Created subgraph #10 with 40 nodes, 117 edges.
    Created subgraph #11 with 42 nodes, 129 edges.
    Created subgraph #12 with 32 nodes, 98 edges.
    Created subgraph #13 with 35 nodes, 98 edges.
    Created subgraph #14 with 38 nodes, 114 edges.
    Created subgraph #15 with 39 nodes, 119 edges.
    Created subgraph #16 with 39 nodes, 102 edges.
    Created subgraph #17 with 36 nodes, 

KeyboardInterrupt: 